# 03 — Neighborhood-based Collaborative Filtering
Phần bổ sung tương thích source mới. Không sửa notebook 01/02/04/thu_nghiem.

**Biểu diễn:** B[u,i]=1 nếu rating ≥4, ngược lại 0. Cosine giữa hai hàng B cho user; giữa hai cột B cho item. Không coi rating thấp là phản hồi yêu thích.

**User-CF:** chọn k user tương tự nhất có similarity dương (loại chính mình). Điểm ứng viên = tổng similarity của các láng giềng thích phim đó.

**Item-CF:** với mỗi phim user thích, chọn k phim gần nhất; cộng similarity từ các phim nguồn đó tới ứng viên. Đây là đồ thị kNN có hướng, không bắt buộc đối xứng.

Tính cosine theo block 128 hàng và giữ k láng giềng; không lưu ma trận N×N đầy đủ. Mã chi tiết trong src/recommenders.py. Điểm để xếp hạng, không phải rating dự đoán; không dùng RMSE/MAE.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src/recommenders.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.recommenders import (load_data, split_validation, Popularity,
                              CollaborativeFiltering, ContentBased, evaluate, fingerprint)
from src.experiments import save_results
train, test, movies = load_data(ROOT)
print('Train:', train.shape, '| Test:', test.shape)
print('Giữ nguyên split 80/20 của notebook 01; dùng movie_id gốc.')
OUT = ROOT / 'reports/tables'
OUT.mkdir(parents=True, exist_ok=True)


Train: (800167, 4) | Test: (200042, 4)
Giữ nguyên split 80/20 của notebook 01; dùng movie_id gốc.


## Quy ước đánh giá chung với 02/04/thu_nghiem
Rating ≥ 4 là tích cực. Candidate là phim trong train, bỏ **mọi phim user đã đánh giá trong train** kể cả rating thấp. Lọc heldout theo user/movie thuộc train rồi chỉ đánh giá user có ít nhất 1 phim tích cực. Precision chia K=10, macro-average theo user; F1 tính từng user rồi lấy trung bình; NDCG dùng relevance nhị phân. Báo số dòng/user được đánh giá và số dòng bị loại.

Evaluator mới phá hòa theo popularity rồi MovieID tăng dần để tái lập; khi nhiều điểm bằng nhau, kết quả có thể khác nhẹ các notebook cũ dùng argsort/sort_values. Baseline được tính lại trong cùng pipeline. Không gọi số liệu notebook cũ là kết quả chạy mới. Đây là random holdout, không phải đánh giá dự đoán tương lai theo thời gian.


## 1. Validation nội bộ, không dùng test chọn k
Tách 10% **từ train 80%** bằng random_state=42. Thử k=10/20/40 cho từng nhánh. Lọc cold-start trên validation theo inner-train. Sau khi chọn k bằng NDCG@10, fit lại **toàn bộ train 80%** và chỉ đánh giá test ở cuối. Không thay đổi train.csv/test.csv, không lấy dữ liệu test để tune.

In [2]:
fit_train, validation = split_validation(train)
rows = []
for mode in ['user', 'item']:
    for neighbors in [10, 20, 40]:
        model = CollaborativeFiltering(mode, neighbors).fit(fit_train)
        metrics, _ = evaluate(model, validation, k=10)
        rows.append({'mode': mode, 'neighbors': neighbors, **metrics})
        print(mode, neighbors, 'NDCG@10 =', round(metrics['NDCG@10'], 6), flush=True)
validation_results = pd.DataFrame(rows)
display(validation_results)
save_results(ROOT, '03_validation_results', validation_results, train, test,
             'Inner split of train only; k chosen by validation NDCG@10.')


user 10 NDCG@10 = 0.131275
user 20 NDCG@10 = 0.146389
user 40 NDCG@10 = 0.156568
item 10 NDCG@10 = 0.141151
item 20 NDCG@10 = 0.137315
item 40 NDCG@10 = 0.135564


,mode,neighbors,Precision@10,Recall@10,F1@10,NDCG@10,evaluated_users,heldout_rows,evaluated_rows,dropped_rows
0,user,10,0.082895,0.143347,0.087448,0.131275,5595,80017,79998,19
1,user,20,0.092225,0.159160,0.097280,0.146389,5595,80017,79998,19
2,user,40,0.098570,0.169329,0.103694,0.156568,5595,80017,79998,19
3,item,10,0.090777,0.143083,0.092580,0.141151,5595,80017,79998,19
4,item,20,0.089062,0.139617,0.090693,0.137315,5595,80017,79998,19
5,item,40,0.088561,0.138892,0.089927,0.135564,5595,80017,79998,19


In [3]:
best_k = {}
for mode in ['user', 'item']:
    best = validation_results[validation_results['mode'] == mode].sort_values(
        ['NDCG@10', 'neighbors'], ascending=[False, True], kind='stable').iloc[0]
    if not np.isfinite(best['NDCG@10']):
        raise ValueError('Validation không có user relevant.')
    best_k[mode] = int(best['neighbors'])
print('Chọn trên validation:', best_k)
config = {'neighbors': best_k, 'train_fingerprint': fingerprint(train)}
(ROOT / 'models').mkdir(exist_ok=True)
(ROOT / 'models/cf_config.json').write_text(json.dumps(config, indent=2))


Chọn trên validation: {'user': 40, 'item': 10}


146

## 2. Fit lại full train, đánh giá test đầy đủ

In [4]:
test_rows = []
for name, model in [
    ('Popularity', Popularity()),
    ('User-CF', CollaborativeFiltering('user', best_k['user'])),
    ('Item-CF', CollaborativeFiltering('item', best_k['item'])),
]:
    model.fit(train)
    metrics, per_user = evaluate(model, test, k=10)
    test_rows.append({'model': name, **metrics})
    per_user.to_csv(OUT / f"03_{name.lower()}_per_user.csv", index=False)
    print(name, metrics, flush=True)
results = pd.DataFrame(test_rows)
display(results)
save_results(ROOT, '03_test_results', results, train, test,
             'Full outer test. CF tuned on inner validation, then refit full train.')


Popularity {'Precision@10': 0.1518018018018018, 'Recall@10': 0.09066595053917328, 'F1@10': 0.0917778988255254, 'NDCG@10': 0.17657381251745477, 'evaluated_users': 5994, 'heldout_rows': 200042, 'evaluated_rows': 200016, 'dropped_rows': 26}
User-CF {'Precision@10': 0.26331331331331337, 'Recall@10': 0.19160627085996035, 'F1@10': 0.17953145002082901, 'NDCG@10': 0.32799553593610625, 'evaluated_users': 5994, 'heldout_rows': 200042, 'evaluated_rows': 200016, 'dropped_rows': 26}
Item-CF {'Precision@10': 0.23997330663997332, 'Recall@10': 0.1602498114597736, 'F1@10': 0.15648767785055417, 'NDCG@10': 0.2941541898185094, 'evaluated_users': 5994, 'heldout_rows': 200042, 'evaluated_rows': 200016, 'dropped_rows': 26}


,model,Precision@10,Recall@10,F1@10,NDCG@10,evaluated_users,heldout_rows,evaluated_rows,dropped_rows
0,Popularity,0.151802,0.090666,0.091778,0.176574,5994,200042,200016,26
1,User-CF,0.263313,0.191606,0.179531,0.327996,5994,200042,200016,26
2,Item-CF,0.239973,0.160250,0.156488,0.294154,5994,200042,200016,26


In [5]:
sample_user = int(train.user_id.iloc[0])
model = CollaborativeFiltering('item', best_k['item']).fit(train)
ranked = pd.DataFrame(model.ranked(model.scores(sample_user), 10, model.seen[sample_user]),
                      columns=['movie_id', 'score'])
display(ranked.merge(movies, on='movie_id', how='left'))


,movie_id,score,title,genres
0,1198,12.730227,Raiders of the Lost Ark (1981),Action|Adventure
1,593,11.049908,"Silence of the Lambs, The (1991)",Drama|Thriller
2,750,7.742230,Dr. Strangelove or: How I Learned to Stop Worr...,Sci-Fi|War
3,1210,6.960273,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
4,919,6.929989,"Wizard of Oz, The (1939)",Adventure|Children's|Drama|Musical
5,1089,6.456892,Reservoir Dogs (1992),Crime|Thriller
6,1036,5.011710,Die Hard (1988),Action|Thriller
7,1234,4.786844,"Sting, The (1973)",Comedy|Crime
8,1094,4.704782,"Crying Game, The (1992)",Drama|Romance|War
9,2396,4.700138,Shakespeare in Love (1998),Comedy|Romance


## Nhận xét 
So sánh Precision/Recall/NDCG với baseline từ bảng vừa chạy. Không mặc định CF luôn tốt hơn. k nhỏ có thể thiếu hàng xóm; k lớn đưa thêm người/phim ít tương tự. Nhị phân hóa bỏ thông tin độ mạnh rating, nhưng khớp mục tiêu positive ranking của BPR. User mới hoặc user không có tín hiệu được gợi ý popularity. Phim mới ngoài catalog train chưa được đánh giá ở protocol này.